# Load Packages

In [ ]:
from glob import glob
from pathlib import Path
import warnings
import numpy as np
from pandas import read_parquet, DataFrame, concat
from scipy.stats import norm
from sklearn.metrics import mean_squared_error, r2_score

import func_gev as gev
import func_preparation as dbf
import func_plotting as dbplt
import func_utils as ut

warnings.filterwarnings("ignore", category=FutureWarning)

# Settings

In [ ]:
path_input = '../input/Annual_max_DCPP_20260112/'
path_results = '../output/gev_analysis/2026-03-20/'

# Import Results

In [ ]:
raw_data = read_parquet(path_results + 'data.parquet')

In [ ]:
[
    results_annual_stat_all, results_nonstat_all, location_geo_info, location_point_info
    ] = ut.import_info_for_regression(path_results)

loading ../output/gev_analysis/2026-03-20//stationary_per_year.pkl
loading ../output/gev_analysis/2026-03-20//nonstationary.pkl
loading ../output/gev_analysis/2026-03-20//LatLon.pkl
loading ../output/gev_analysis/2026-03-20//location_info.pkl


In [ ]:
results_stat_all = ut.import_pickle_data(path_results, 'stationary.pkl')

loading ../output/gev_analysis/2026-03-20//stationary.pkl


In [ ]:
model_comparison_all = ut.import_pickle_data(path_results, 'model_comparison.pkl')

loading ../output/gev_analysis/2026-03-20//model_comparison.pkl


In [ ]:
return_levels = read_parquet(path_results + 'return_levels.parquet')

# Model Comparison · stationary vs non-stationary preference

In [ ]:
model_comparison_pvalue = [comparison['LRT']['p_value'] for comparison in model_comparison_all.values()]
df_stat_model_pvalue = DataFrame(model_comparison_pvalue)
df_stat_model_pvalue[df_stat_model_pvalue[0] < 0.05].count()

In [ ]:
model_comparison_interpret = [comparison['LRT']['interpretation'] for comparison in model_comparison_all.values()]

df_stat_model_comp = DataFrame(model_comparison_interpret).value_counts()
df_stat_model_comp

In [ ]:
print(
    'Non-stationary GEV was preferred at '
    f'{df_stat_model_comp.filter(like='non-stationary', axis=0).values[0] / len(model_comparison_interpret) * 100:.2f}% ' 
    'of sites'
    ) 

In [ ]:
for siteID, nonstat in results_nonstat_all.items():
    if nonstat['cov'] is None:
        print('covariance is None for {}')

df_nonstat_cov = [nonstat['cov'] for nonstat in results_nonstat_all.values()]


In [ ]:
results_stat_all[0].get('cov')

In [ ]:
ls_nocov_stat = []
for siteID, stat in results_stat_all.items():
    if stat.get('cov') is None:
        ls_nocov_stat.append(siteID)

#df_stat_cov = [stat['cov'] for stat in results_stat_all.values()]
len(ls_nocov_stat)

# Location Trend Overview

## Non-stationary

In [ ]:
df_results_nonstat = DataFrame([results_nonstat['params_hat'] for results_nonstat in results_nonstat_all.values()])
df_results_nonstat.columns = ['loc', 'loc_trend','scale', 'shape']

df_results_nonstat['loc_trend_mm/yr'] = df_results_nonstat['loc_trend']*1000

df_results_nonstat

,loc,loc_trend,scale,shape,loc_trend_mm/yr
0,0.125558,-0.000012,0.031529,-0.110467,-0.011510
1,0.125707,-0.000012,0.031585,-0.110656,-0.011870
2,0.125430,-0.000011,0.031439,-0.110925,-0.010767
3,0.125618,-0.000012,0.031538,-0.110734,-0.011880
4,0.131225,-0.000062,0.034860,-0.111517,-0.062174
...,...,...,...,...,...
9584,0.235585,0.000006,0.044659,-0.207715,0.005627
9585,0.294627,-0.000065,0.047133,-0.198150,-0.065097
9586,0.294834,-0.000090,0.062643,-0.206983,-0.090034
9587,0.297700,-0.000088,0.063634,-0.214620,-0.088083


##### outlier removal

In [ ]:
label_para = 'loc_trend_mm/yr'

para_nonstat_outlier = ut.mark_outliers_zmethod(df_results_nonstat, label_col=label_para, threshold=3)
para_nonstat_outlier = para_nonstat_outlier[para_nonstat_outlier.outliers == False]

if label_para == 'loc_trend' or label_para == 'scale':
    para_nonstat_outlier = para_nonstat_outlier*1000
    
print(f'median: {para_nonstat_outlier[label_para].median():.3f}')
print(f"min:\t{para_nonstat_outlier[label_para].describe()['min']:.3f}")
print(f"max:\t{para_nonstat_outlier[label_para].describe()['max']:.3f}")
print(f"STD:\t{para_nonstat_outlier[label_para].describe()['std']:.3f}")

para_nonstat_outlier.describe()

Marked 89 outliers using modified Z-score method
median: -0.049
min:	-0.622
max:	0.584
STD:	0.171


,loc,loc_trend,scale,shape,loc_trend_mm/yr
count,9500.000000,9500.000000,9500.000000,9500.000000,9500.000000
mean,0.549932,-0.000014,0.104518,-0.154218,-0.014430
std,0.374646,0.000171,0.067913,0.071881,0.170745
min,0.117837,-0.000622,0.025183,-0.563239,-0.621920
25%,0.238779,-0.000106,0.047247,-0.200606,-0.105663
50%,0.418789,-0.000049,0.079597,-0.164699,-0.049454
75%,0.810060,0.000060,0.152161,-0.118420,0.059747
max,2.152442,0.000584,0.416195,0.150610,0.584140


### statistics

In [ ]:
df_mu1_pos = df_results_nonstat[df_results_nonstat.loc_trend >0].loc_trend.count() / df_results_nonstat.shape[0]* 100
df_mu1_pos_outliers = para_nonstat_outlier[para_nonstat_outlier.loc_trend >0].loc_trend.count() / para_nonstat_outlier.shape[0]* 100

df_mu1_pos, df_mu1_pos_outliers

(np.float64(35.29043695901554), np.float64(35.26315789473684))

In [ ]:
site_loc_trend_max = df_results_nonstat.loc_trend.idxmax()
location_point_info[site_loc_trend_max], location_geo_info[site_loc_trend_max]

('Arnside England GB',
 (np.float64(54.19770057729435), np.float64(-2.8627750910386593)))

In [ ]:
site_loc_trend_max_outlier = para_nonstat_outlier.loc_trend.idxmax()
location_point_info[site_loc_trend_max_outlier], location_geo_info[site_loc_trend_max_outlier]

('Montmartin-sur-Mer Lower Normandy FR',
 (np.float64(48.94675819157106), np.float64(-1.5648041570045255)))

In [ ]:
print(
    f'Positive μ₁ trends were identified at {df_mu1_pos:.2f}% of sites ({df_mu1_pos_outliers:.2f}% considering outliers),'
    f'\nwith the strongest trends concentrated along {location_point_info[site_loc_trend_max]}'
    )

Positive μ₁ trends were identified at 35.29% of sites (35.26% considering outliers),
with the strongest trends concentrated along Arnside England GB


##  Annual-stationary

In [ ]:
df_mu1_astat_all = DataFrame(
    [astat['mu_trend']['mu1'] for  astat in results_annual_stat_all.values()], 
    index=results_annual_stat_all.keys(), columns=['loc_trend']
    )
df_mu1_astat_all['loc_trend_mm/yr'] = df_mu1_astat_all['loc_trend']*1000
df_mu1_astat_all

,loc_trend,loc_trend_mm/yr
0,-0.000077,-0.076743
1,-0.000058,-0.057600
2,-0.000056,-0.056471
3,-0.000066,-0.066158
4,0.000109,0.109255
...,...,...
9584,-0.000443,-0.442955
9585,-0.000212,-0.212398
9586,-0.000094,-0.094383
9587,-0.000224,-0.223714


#### outlier removal

In [ ]:
label_para = 'loc_trend_mm/yr'

para_astat_outlier = ut.mark_outliers_zmethod(df_mu1_astat_all, label_col=label_para, threshold=3)
para_astat_outlier = para_astat_outlier[para_astat_outlier.outliers == False]

if label_para == 'loc_trend' or label_para == 'scale':
    para_astat_outlier = para_astat_outlier*1000
    
print(f'median: {para_astat_outlier[label_para].median():.3f}')
print(f"min:\t{para_astat_outlier[label_para].describe()['min']:.3f}")
print(f"max:\t{para_astat_outlier[label_para].describe()['max']:.3f}")
print(f"STD:\t{para_astat_outlier[label_para].describe()['std']:.3f}")

para_astat_outlier.describe()

Marked 172 outliers using modified Z-score method
median: -0.050
min:	-1.528
max:	1.626
STD:	0.431


,loc_trend,loc_trend_mm/yr
count,9417.000000,9417.000000
mean,0.000053,0.053023
std,0.000431,0.430832
min,-0.001528,-1.528384
25%,-0.000159,-0.159371
50%,-0.000050,-0.049927
75%,0.000235,0.235303
max,0.001626,1.625559


### statistics

In [ ]:
df_mu1_astat_pos = para_astat_outlier[para_astat_outlier.loc_trend >0].loc_trend.count() / para_astat_outlier.shape[0]* 100
df_mu1_astat_pos_outliers = para_astat_outlier[para_astat_outlier.loc_trend >0].loc_trend.count() / para_astat_outlier.shape[0]* 100

df_mu1_astat_pos, df_mu1_astat_pos_outliers

(np.float64(43.060422639906555), np.float64(43.060422639906555))

## Comparison

In [ ]:
mu1_astat_all = df_mu1_astat_all['loc_trend_mm/yr'].values
mu1_ns_all = df_results_nonstat['loc_trend_mm/yr'].values

rmse = np.sqrt(mean_squared_error(mu1_ns_all, mu1_astat_all))
r2 = r2_score(mu1_ns_all, mu1_astat_all)

print(f'RMSE: {rmse:.4f}')
print(f'R²: {r2:.4f}')
print(f'Mean Absolute Error (non-stationary): {np.mean(np.abs(mu1_ns_all)):.4f} mm/year')
print(f'Mean Absolute Error (annual-stationary): {np.mean(np.abs(mu1_astat_all)):.4f} mm/year')

RMSE: 0.4722
R²: -4.4298
Mean Absolute Error (non-stationary): 0.1379 mm/year
Mean Absolute Error (annual-stationary): 0.3370 mm/year


In [ ]:
df_comparison = concat([
    para_nonstat_outlier['loc_trend_mm/yr'], para_astat_outlier['loc_trend_mm/yr']
    ], axis=1).dropna()
df_comparison.columns = ['loc_trend_nonstat', 'loc_trend_astat']

df_comparison.describe()

,loc_trend_nonstat,loc_trend_astat
count,9351.000000,9351.000000
mean,-0.013788,0.052333
std,0.169434,0.427263
min,-0.621920,-1.528384
25%,-0.105227,-0.159200
50%,-0.049638,-0.050146
75%,0.058927,0.233289
max,0.584140,1.625559


In [ ]:
mu1_astat_all = df_comparison.loc_trend_astat.values
mu1_ns_all = df_comparison.loc_trend_nonstat.values

rmse = np.sqrt(mean_squared_error(mu1_ns_all, mu1_astat_all))
r2 = r2_score(mu1_ns_all, mu1_astat_all)

print(f'RMSE: {rmse:.3f}')
print(f'R²: {r2:.3f}')
print(f'Mean Absolute Error (non-stationary): {np.median(np.abs(mu1_ns_all)):.4f} mm/year')
print(f'Mean Absolute Error (annual-stationary): {np.median(np.abs(mu1_astat_all)):.4f} mm/year')

RMSE: 0.375
R²: -3.894
Mean Absolute Error (non-stationary): 0.0965 mm/year
Mean Absolute Error (annual-stationary): 0.1818 mm/year


# Return Period Overview

In [ ]:
return_levels_50yr = return_levels[return_levels.return_period == 50]
return_levels_50yr = return_levels_50yr[return_levels_50yr.t_eval == 1961]

# a 50year event in 1961 
return_levels_50yr

,z_T,lower,upper,t_eval,return_period,model,location_id
12,0.225520,0.216033,0.235006,1961,50,stationary,0
13,0.225524,0.183529,0.267518,1961,50,nonstationary,0
42,0.225803,0.216326,0.235281,1961,50,stationary,1
43,0.225816,0.183836,0.267796,1961,50,nonstationary,1
72,0.225023,0.215550,0.234496,1961,50,stationary,2
...,...,...,...,...,...,...,...
287593,0.462729,0.365028,0.560431,1961,50,nonstationary,9586
287622,0.497270,0.450670,0.543869,1961,50,stationary,9587
287623,0.466062,0.369987,0.562137,1961,50,nonstationary,9587
287652,0.501121,0.453610,0.548632,1961,50,stationary,9588


In [ ]:
return_levels_50yr_stat = return_levels_50yr[return_levels_50yr.model == 'stationary'][['z_T','lower', 'upper']].median()
return_levels_50yr_stat_m = return_levels_50yr_stat*1000 # meter

In [ ]:
return_levels_50yr_nstat = return_levels_50yr[return_levels_50yr.model == 'nonstationary'][['z_T','lower', 'upper']].median()
return_levels_50yr_nstat_m = return_levels_50yr_nstat*1000 # meter

In [ ]:
return_levels_50yr_stat_m

In [ ]:
print(
    'The median 50-year return level across all sites was\n',
    f'stationary approach: {return_levels_50yr_stat_m.z_T:.2f}m (95% CI: {return_levels_50yr_stat_m.lower:.2f}m - {return_levels_50yr_stat_m.upper:.2f}m)\n',
    f'non-stationary approach: {return_levels_50yr_nstat_m.z_T:.2f}m (95% CI: {return_levels_50yr_nstat_m.lower:.2f}m - {return_levels_50yr_nstat_m.upper:.2f}m)\n',  
)

# Return Levels

In [ ]:
t_eval = 1961

for approach in ['stationary', 'nonstationary']:
    print(f'Summary {approach} approach:')
    for return_period in [10, 50, 100]:
        return_level_approach = return_levels[return_levels.model == approach]

        return_level_approach_teval = return_level_approach[return_level_approach.t_eval == t_eval]
        df = return_level_approach_teval[return_level_approach_teval['return_period'] == return_period]

        q25 = df[['z_T', 'lower', 'upper']].describe().loc['25%']
        q75 = df[['z_T', 'lower', 'upper']].describe().loc['75%']
        median_ = df[['z_T', 'lower', 'upper']].median().T.loc['z_T']

        iqr = q75 - q25
        ci_width = (df['upper'] - df['lower']).median()
            
        print(
            f"\tT={return_period}: median = {median_:.3f} m "
            f"| iqr = {iqr['z_T']:.3f} m "
            f"| median 90%CI width={ci_width:.3f} m"
            )
        

Summary stationary approach:
	T=10: median = 0.585 m | iqr = 0.794 m | median 90%CI width=0.030 m
	T=50: median = 0.689 m | iqr = 0.910 m | median 90%CI width=0.034 m
	T=100: median = 0.730 m | iqr = 0.952 m | median 90%CI width=0.036 m
Summary nonstationary approach:
	T=10: median = 0.582 m | iqr = 0.787 m | median 90%CI width=0.063 m
	T=50: median = 0.679 m | iqr = 0.885 m | median 90%CI width=0.094 m
	T=100: median = 0.716 m | iqr = 0.915 m | median 90%CI width=0.106 m


# Fit Parameters

### stationary

In [ ]:
para_stat = concat([DataFrame([[
    result_stat['location'], result_stat['location_std'],
    result_stat['scale'], result_stat['scale_std'],
    result_stat['shape'], result_stat['shape_std']]
], index=[siteID]) for siteID, result_stat in results_stat_all.items()])

para_stat.columns = ['loc', 'loc_std', 'scale', 'scale_std', 'shape', 'shape_std']

#### outlier removal and description

In [ ]:
label_para = 'scale'

para_stat_outlier = ut.mark_outliers_zmethod(para_stat, label_col=label_para, threshold=3)
para_stat_outlier = para_stat_outlier[para_stat_outlier.outliers == False]

if label_para == 'loc_trend' or label_para == 'scale':
    para_stat_outlier = para_stat_outlier*1000
    
print(f'median: {para_stat_outlier[label_para].median():.3f}')
print(f"min:\t{para_stat_outlier[label_para].describe()['min']:.3f}")
print(f"max:\t{para_stat_outlier[label_para].describe()['max']:.3f}")
print(f"STD:\t{para_stat_outlier[label_para].describe()['std']:.3f}")

para_stat_outlier.describe()

In [ ]:
para_stat[para_stat['shape'] < 0]['shape'].count() / para_stat.shape[0] * 100

## non-stationary

In [ ]:
para_nonstat = concat([
    DataFrame(nonstat['params_hat'], columns=[siteID]).T 
    for siteID, nonstat in results_nonstat_all.items()
    ])
para_nonstat.columns = ['loc', 'loc_trend', 'scale', 'shape']

### outlier removal and description

In [ ]:
label_para = 'loc_trend'

para_nonstat_outlier = ut.mark_outliers_zmethod(para_nonstat, label_col=label_para, threshold=3)
para_nonstat_outlier = para_nonstat_outlier[para_nonstat_outlier.outliers == False]

if label_para == 'loc_trend' or label_para == 'scale':
    para_nonstat_outlier = para_nonstat_outlier*1000
    
print(f'median: {para_nonstat_outlier[label_para].median():.3f}')
print(f"min:\t{para_nonstat_outlier[label_para].describe()['min']:.3f}")
print(f"max:\t{para_nonstat_outlier[label_para].describe()['max']:.3f}")
print(f"STD:\t{para_nonstat_outlier[label_para].describe()['std']:.3f}")

para_nonstat_outlier.describe()

Marked 89 outliers using modified Z-score method
median: -0.049
min:	-0.622
max:	0.584
STD:	0.171


,loc,loc_trend,scale,shape,outliers
count,9500.000000,9500.000000,9500.000000,9500.000000,9500.0
mean,549.932500,-0.014430,104.517524,-154.217804,0.0
std,374.646488,0.170745,67.912788,71.880918,0.0
min,117.837234,-0.621920,25.183295,-563.238921,0.0
25%,238.778592,-0.105663,47.246678,-200.606428,0.0
50%,418.789155,-0.049454,79.596788,-164.699500,0.0
75%,810.059600,0.059747,152.161302,-118.420352,0.0
max,2152.441946,0.584140,416.194970,150.609580,0.0


### statistics

In [ ]:
n_total_sites = para_nonstat_outlier.shape[0]

In [ ]:
para_nonstat_outlier[para_nonstat_outlier['shape'] < 0]['shape'].count() / n_total_sites * 100

In [ ]:
para_nonstat_outlier[para_nonstat_outlier.loc_trend > 0].loc_trend.count() / n_total_sites * 100

np.float64(35.26315789473684)

In [ ]:
# Wald statistic to p-value in location trend (two-sided)
z_stat = para_nonstat_outlier.loc_trend / para_nonstat_outlier.loc_trend.std()          
pvalue = 2 * (1 - norm.cdf(abs(z_stat))) 

(pvalue < 0.05).mean() * 100

np.float64(6.747368421052631)

In [ ]:
location_geo_info[para_nonstat_outlier.loc_trend.idxmax()], location_point_info[para_nonstat_outlier.loc_trend.idxmax()]

((np.float64(48.94675819157106), np.float64(-1.5648041570045255)),
 'Montmartin-sur-Mer Lower Normandy FR')

In [ ]:
location_geo_info[para_nonstat_outlier.loc_trend.idxmin()], location_point_info[para_nonstat_outlier.loc_trend.idxmin()]

((np.float64(55.46231382844708), np.float64(-1.5884571683002715)),
 'Embleton England GB')

In [ ]:
para_nonstat_outlier.loc_trend.max(), para_nonstat_outlier.loc_trend.min()

(np.float64(0.5841403769113266), np.float64(-0.6219200095859199))

## annual-stationary

In [ ]:
para_astat = concat([
    concat([astat['annual_mle'][['location', 'scale', 'shape']].median() 
            for astat in results_annual_stat_all.values()], axis=1).T,
    DataFrame([
        astat['mu_trend']['mu1'] 
        for astat in results_annual_stat_all.values()], columns=['mu1'])
    ], axis=1)

para_astat.columns = ['loc', 'scale', 'shape', 'loc_trend']
para_astat

,loc,scale,shape,loc_trend
0,0.125857,0.030841,-0.108257,-0.000077
1,0.126116,0.030694,-0.113056,-0.000058
2,0.125699,0.030421,-0.113170,-0.000056
3,0.125948,0.030727,-0.110627,-0.000066
4,0.131861,0.034328,-0.119333,0.000109
...,...,...,...,...
9584,0.235528,0.044031,-0.242287,-0.000443
9585,0.298117,0.043282,-0.260169,-0.000212
9586,0.300370,0.057717,-0.271753,-0.000094
9587,0.302652,0.057762,-0.300442,-0.000224


### outlier removal and description

In [ ]:
label_para = 'loc_trend'

para_astat_outlier = ut.mark_outliers_zmethod(para_astat, label_col=label_para, threshold=3)
para_astat_outlier = para_astat_outlier[para_astat_outlier.outliers == False]

if label_para == 'loc_trend' or label_para == 'scale':
    para_astat_outlier = para_astat_outlier*1000
    
print(f'median: {para_astat_outlier[label_para].median():.3f}')
print(f"min:\t{para_astat_outlier[label_para].describe()['min']:.3f}")
print(f"max:\t{para_astat_outlier[label_para].describe()['max']:.3f}")
print(f"STD:\t{para_astat_outlier[label_para].describe()['std']:.3f}")

para_astat_outlier.describe()

Marked 172 outliers using modified Z-score method
median: -0.050
min:	-1.528
max:	1.626
STD:	0.431


,loc,scale,shape,loc_trend,outliers
count,9417.000000,9417.000000,9417.000000,9417.000000,9417.0
mean,545.380374,97.730990,-152.498177,0.053023,0.0
std,373.927972,62.444558,110.620723,0.430832,0.0
min,118.115071,23.861080,-1243.631233,-1.528384,0.0
25%,238.552401,46.484234,-211.114704,-0.159371,0.0
50%,414.615578,74.794794,-154.172661,-0.049927,0.0
75%,796.895071,139.400161,-81.108905,0.235303,0.0
max,2216.931746,420.463788,213.740709,1.625559,0.0


### statistics

In [ ]:
n_total_sites_astat = para_astat_outlier.shape[0]

In [ ]:
para_astat_outlier[para_astat_outlier['shape'] < 0]['shape'].count() / n_total_sites_astat * 100

np.float64(93.28873314218966)

In [ ]:
para_astat_outlier[para_astat_outlier.loc_trend > 0].loc_trend.count() / n_total_sites_astat * 100

np.float64(43.060422639906555)

In [ ]:
# Wald statistic to p-value in location trend (two-sided)
z_stat = para_astat_outlier.loc_trend / para_astat_outlier.loc_trend.std()          
pvalue = 2 * (1 - norm.cdf(abs(z_stat))) 

(pvalue < 0.05).mean() * 100

np.float64(8.187320802803441)

In [ ]:
print('maximum location trend:',
    location_geo_info[para_astat_outlier.loc_trend.idxmax()], 
    location_point_info[para_astat_outlier.loc_trend.idxmax()], 
    para_astat_outlier.loc_trend.max()
)

maximum location trend: (np.float64(56.46405982078649), np.float64(8.124438719530673)) Harboore Central Jutland DK 1.6255592390601228


In [ ]:
print('minimum location trend:',
    location_geo_info[para_astat_outlier.loc_trend.idxmin()], 
    location_point_info[para_astat_outlier.loc_trend.idxmin()], 
    para_astat_outlier.loc_trend.min()
)

minimum location trend: (np.float64(51.869324800488485), np.float64(1.2944004765110506)) Walton-on-the-Naze England GB -1.5283835664864223
